In [1]:
import os
import time
from pymmcore_plus import CMMCorePlus
from PyQt5.QtWidgets import QApplication, QMainWindow, QSlider, QLineEdit, QPushButton, QLabel, QComboBox, QWidget, QGridLayout
from PyQt5.QtGui import QPixmap, QImage
from PyQt5.QtCore import QTimer, pyqtSignal, pyqtSlot, Qt, QThread
from PyQt5 import QtGui, QtCore, QtWidgets
import qimage2ndarray
import numpy as np
import sys
from PIL import Image
import re
from skimage import exposure


class QToggleButton(QPushButton):
    def __init__(self, text='', parent=None):
        super().__init__(text, parent)
        
        # Ensure the button is checkable
        self.setCheckable(True)
        
        # Set the button to be checked by default
        self.setChecked(False)
        self.setStyleSheet("background-color: #bb283a;")



Microscope = {}
Microscope['mmc'] = CMMCorePlus.instance()
Microscope['mmc'].loadSystemConfiguration("C:\\MATLAB Microscope\\AmoghMMConfig.cfg")


class VideoThread(QThread):
    change_pixmap_signal = pyqtSignal(np.ndarray)

    def __init__(self):
        super().__init__()
        self._run_flag = True
        self.mmc = Microscope['mmc']
        self.camera = self.mmc.getCameraDevice()
        self.DIAshutter = self.mmc.getShutterDevice()
        self.focus = self.mmc.getFocusDevice()
        self.stage = self.mmc.getXYStageDevice()
        self.PFS = self.mmc.getAutoFocusDevice()
        self.EPIshutter = 'TIEpiShutter'
        self.DIAlamp = 'TIDiaLamp'
        self.scope = 'TIScope'
        self.zoom = 'TINosePiece'
        self.filter = 'TIFilterBlock1'
        self.lightpath = 'TILightPath'
        self.PFS_offset = 'TIPFSOffset'
        self.core = 'Core'
        self.camerapath = '3-Right100'

    def run(self):
        # capture from web cam
        self.mmc.setProperty(self.DIAshutter,'State',0)
        self.mmc.setProperty(self.EPIshutter,'State',0)
        self.mmc.setProperty(self.lightpath,'Label',self.camerapath)
        self.mmc.initializeCircularBuffer()
        self.mmc.prepareSequenceAcquisition(self.camera)
        self.mmc.waitForDevice(self.DIAshutter)
        self.mmc.waitForDevice(self.camera)
        interval = 100 # image per X ms
        self.mmc.startContinuousSequenceAcquisition(interval)
        self.Sequencing = self.mmc.isSequenceRunning() # Outputs 1 to indicate video sequence is running
        while self._run_flag and self.Sequencing:
            if self.mmc.getRemainingImageCount() > 0:
                liveimage = self.mmc.getLastImage()  #Retrieves last image taken
                image_width = self.mmc.getImageWidth()
                image_height = self.mmc.getImageHeight()
                pixelType = np.uint16
                self.final_image = self.convert_raw_np(liveimage, image_width, image_height, pixelType)
                self.change_pixmap_signal.emit(self.final_image)

        self.mmc.stopSequenceAcquisition(self.camera) #Stop acquisition and shut down capture system
        self.mmc.clearCircularBuffer()
    

    def convert_raw_np(self, raw_img, img_width, img_height, pixel_Type):
        """Convert from an raw image to QPixmap"""
        # Assuming rawImage is a byte array
        rawImage = np.frombuffer(raw_img, dtype= pixel_Type)
        rawImage = rawImage.reshape((img_height, img_width)).T
        # Now assuming rawImage is a numpy array
        self.adjusted_image = exposure.rescale_intensity(rawImage)
        return self.adjusted_image

    def stop(self):
        # Sets run flag to False and waits for thread to finish
        self._run_flag = False
        self.wait()



class MicroscopeControlGUI(QMainWindow):
    def __init__(self):
        super().__init__()
        self.initUI()

    def initUI(self):
        # Change working directory
        os.chdir('C:/Users/Cell Culture Scope/Documents/MATLAB')

        # Delete all timers (if any in Micro-Manager context)
        # Note: Python doesn't have a direct equivalent to MATLAB's timerfindall and delete
        # You'll need to implement your own logic here if necessary

        # Set dark mode palette
        dark_palette = QtGui.QPalette()
        dark_palette.setColor(QtGui.QPalette.Window, QtGui.QColor(53, 53, 53))
        dark_palette.setColor(QtGui.QPalette.WindowText, QtCore.Qt.white)
        dark_palette.setColor(QtGui.QPalette.Base, QtGui.QColor(25, 25, 25))
        dark_palette.setColor(QtGui.QPalette.AlternateBase, QtGui.QColor(53, 53, 53))
        dark_palette.setColor(QtGui.QPalette.ToolTipBase, QtCore.Qt.white)
        dark_palette.setColor(QtGui.QPalette.ToolTipText, QtCore.Qt.white)
        dark_palette.setColor(QtGui.QPalette.Text, QtCore.Qt.white)
        dark_palette.setColor(QtGui.QPalette.Button, QtGui.QColor(53, 53, 53))
        dark_palette.setColor(QtGui.QPalette.ButtonText, QtCore.Qt.white)
        dark_palette.setColor(QtGui.QPalette.BrightText, QtCore.Qt.red)
        dark_palette.setColor(QtGui.QPalette.Link, QtGui.QColor(42, 130, 218))
        dark_palette.setColor(QtGui.QPalette.Highlight, QtGui.QColor(42, 130, 218))
        dark_palette.setColor(QtGui.QPalette.HighlightedText, QtCore.Qt.black)
        
        self.setPalette(dark_palette)

        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        self.layout = QGridLayout()
        central_widget.setLayout(self.layout)

        # Slider Initialization
        """self.dialampslider = QSlider(self)
        self.dialampslider.setMinimum(1)
        self.dialampslider.setMaximum(24)
        self.dialampslider.setValue(2)
        steps_dia = [1/50, 24/50]  # Step values
        self.dialampslider.setSingleStep(int(steps_dia[0]*100))"""

        """self.EMslider = QSlider(self)
        self.EMslider.setMinimum(25)
        self.EMslider.setMaximum(51)
        self.EMslider.setSingleStep(1)"""

        # Micro-Manager Initialization
        self.mmc = Microscope['mmc']
        #self.mmc.loadSystemConfiguration("C:\\MATLAB Microscope\\AmoghMMConfig.cfg")

        Devices = self.mmc.getLoadedDevices()
        Devices_list = [Devices[i] for i in range(len(Devices))]

        self.camera = self.mmc.getCameraDevice()
        self.DIAshutter = self.mmc.getShutterDevice()
        self.focus = self.mmc.getFocusDevice()
        self.stage = self.mmc.getXYStageDevice()
        self.PFS = self.mmc.getAutoFocusDevice()
        self.EPIshutter = 'TIEpiShutter'
        self.DIAlamp = 'TIDiaLamp'
        self.scope = 'TIScope'
        self.zoom = 'TINosePiece'
        self.filter = 'TIFilterBlock1'
        self.lightpath = 'TILightPath'
        self.PFS_offset = 'TIPFSOffset'
        self.core = 'Core'

        CamProperties = self.mmc.getDevicePropertyNames(self.camera)
        self.CamProperties_list = [CamProperties[i] for i in range(len(CamProperties))]

        ScopeProperties = self.mmc.getDevicePropertyNames(self.scope)
        self.ScopeProperties_list = [ScopeProperties[i] for i in range(len(ScopeProperties))]

        DIALampProperties = self.mmc.getDevicePropertyNames(self.DIAlamp)
        self.DIALampProperties_list = [DIALampProperties[i] for i in range(len(DIALampProperties))]

        ZoomProperties = self.mmc.getDevicePropertyNames(self.zoom)
        self.ZoomProperties_list = [ZoomProperties[i] for i in range(len(ZoomProperties))]
        

        self.zoom4x = '1-(Achromat) 4x NA 0.10 Dry'
        self.zoom10x = '2-(Achromat) 10x NA 0.25 Dry'
        self.zoom20x = '3-(Achromat) 20x NA 0.40 Dry'
        self.zoom40x = '4-S Plan Fluor 40x NA 0.60 Dry'
        self.zoom60x = '5-Plan Apo 60x NA 1.40 Oil'
        self.zoomempty = '6-Unknown'
        

        FilterProperties = self.mmc.getDevicePropertyNames(self.filter)
        self.FilterProperties_list = [FilterProperties[i] for i in range(len(FilterProperties))]

        self.filter_1 = '1-Cy5'
        self.filter_2 = '2-Cy3'
        self.filter_3 = '3-TRITC'
        self.filter_4 = '4-FITC'
        self.filter_5 = '5-DAPI'
        self.filter_6 = '6-DIA'

        CoreProperties = self.mmc.getDevicePropertyNames(self.core)
        self.CoreProperties_list = [CoreProperties[i] for i in range(len(CoreProperties))]

        DIAShutterProperties = self.mmc.getDevicePropertyNames(self.DIAshutter)
        self.DIAShutterProperties_list = [DIAShutterProperties[i] for i in range(len(DIAShutterProperties))]

        EPIShutterProperties = self.mmc.getDevicePropertyNames(self.EPIshutter)
        self.EPIShutterProperties_list = [EPIShutterProperties[i] for i in range(len(EPIShutterProperties))]

        StageProperties = self.mmc.getDevicePropertyNames(self.stage)
        self.StageProperties_list = [StageProperties[i] for i in range(len(StageProperties))]

        LightPathProperties = self.mmc.getDevicePropertyNames(self.lightpath)
        self.LightPathProperties_list = [LightPathProperties[i] for i in range(len(LightPathProperties))]

        self.eyepath = '1-Eye100'
        self.camerapath = '3-Right100'

        # Initialization for the stage controls
        """self.xyfastlight.setChecked(True)
        self.xymediumlight.setChecked(False)
        self.xyslowlight.setChecked(False)
        self.zfastlight.setChecked(True)
        self.zmediumlight.setChecked(False)
        self.zslowlight.setChecked(False)"""

        # Camera Initialization
        self.mmc.setProperty(self.camera, 'CCDTemperatureSetPoint', -70)
    
        self.mmc.setProperty(self.camera, 'Exposure', 50)

        self.mmc.setProperty(self.camera, 'Binning', 2)
        self.mmc.setProperty(self.lightpath, 'Label', self.eyepath)

        self.eyepathlight = QPushButton("Path to Eye")
        self.eyepathlight.setChecked(True)
        self.eyepathlight.clicked.connect(self.PathtoCamera)
        
        self.camerapathlight = QPushButton("Path to Camera")
        self.camerapathlight.setChecked(True)
        self.camerapathlight.clicked.connect(self.PathtoCamera)

        

        self.mmc.setProperty(self.camera, 'Gain', 10)
        #self.gaintext = QLineEdit(self)
        #self.gaintext.setText('10')

        self.mmc.setProperty(self.camera, 'Pre-Amp-Gain', '5.1x')
        #self.EMslider.setValue(51)
        #self.EMtext = QLineEdit(self)
        #self.EMtext.setText('5.1x')

        self.mmc.setProperty(self.camera, 'PixelType', '16bit')

        # Zoom Initialization
        self.Zoom_list = QComboBox()
        self.Zoom_list.addItems([self.zoom4x, self.zoom10x, self.zoom20x, self.zoom40x, self.zoom60x, self.zoomempty])
        self.Zoom_list.currentTextChanged.connect(self.Set_zoom)
        #self.layout.addWidget(self.Zoom_list)
        zoom_ini = self.mmc.getProperty(self.zoom, 'Label')
        #print(zoom_ini)
        """if zoom_ini == self.zoom4x:
            self.zoomlight4x.setChecked(True)
        elif zoom_ini == self.zoom10x:
            self.zoomlight10x.setChecked(True)
        elif zoom_ini == self.zoom20x:
            self.zoomlight20x.setChecked(True)
        elif zoom_ini == self.zoom40x:
            self.zoomlight40x.setChecked(True)
        elif zoom_ini == self.zoom60x:
            self.zoomlight60x.setChecked(True)"""

        # Light Path Initialization
        self.mmc.setProperty(self.lightpath, 'Label', self.eyepath)
        self.eyepathlight.setChecked(True)
        self.camerapathlight.setChecked(False)

        

        # DIA Lamp Initialization
        self.mmc.setProperty(self.DIAlamp, 'ComputerControl', 'On')
        """self.dialampmanuallight = QPushButton(self)
        self.dialampmanuallight.setChecked(False)
        self.dialampsoftwarelight = QPushButton(self)
        self.dialampsoftwarelight.setChecked(True)"""

        self.mmc.setProperty(self.DIAlamp, 'Intensity', 4)
        self.mmc.setProperty(self.DIAlamp, 'State', 0)
        self.dialamponlight = QToggleButton()
        self.dialamponlight.setText("DIA Lamp")
        self.dialamponlight.clicked.connect(self.DIAlamp_ON)
        """self.dialampvalue = QLineEdit(self)
        self.dialampvalue.setText('2')"""

        # Shutter Initialization
        self.mmc.setProperty(self.DIAshutter, 'State', 0)
        self.mmc.setProperty(self.EPIshutter, 'State', 0)

        #DIA Shutter
        self.DIAshutterbutton = QToggleButton("DIA Shutter")
        self.DIAshutterbutton.clicked.connect(self.toggle_DIA_shutter)

        #EPI Shutter
        self.EPIshutterbutton = QToggleButton("EPI Shutter")
        self.EPIshutterbutton.clicked.connect(self.toggle_EPI_shutter)



            #Adding all shutter buttons
        self.layout.addWidget(self.dialamponlight, 0, 0)
        self.layout.addWidget(self.DIAshutterbutton, 0, 1)
        self.layout.addWidget(self.EPIshutterbutton, 0, 2)
        self.layout.addWidget(self.Zoom_list, 1, 0, 1, 3)

                                                            # Change Filters
        
        filter_buttons_layout = QtWidgets.QHBoxLayout()

        # Filter Initialization
        filter_ini = self.mmc.getProperty(self.filter, 'Label')
        """if filter_ini == self.filter_Cy5_1:
            self.cy5filterlight.setChecked(True)
        elif filter_ini == self.filter_Cy3_2:
            self.cy3filterlight.setChecked(True)
        elif filter_ini == self.filter_DIA_6:
            self.DIAfilterlight.setChecked(True)"""
        
        filter_buttons = {
            self.filter_1: QPushButton(self.filter_1),
            self.filter_2: QPushButton(self.filter_2),
            self.filter_3: QPushButton(self.filter_3),
            self.filter_4: QPushButton(self.filter_4),
            self.filter_5: QPushButton(self.filter_5),
            self.filter_6: QPushButton(self.filter_6)
        }
        
        for filter_name, button in filter_buttons.items():
            button.clicked.connect(lambda checked, fn=filter_name: self.change_filter(fn))
            #self.layout.addWidget(button)
        
        row = 2
        col = 0
        for button in filter_buttons.values():
            self.layout.addWidget(button, row, col)
            col += 1
            if col > 2:
                col = 0
                row += 1

        
        


                                                            # Stage Slots

            # Reading stage position
        self.xcoord = self.mmc.getXPosition(self.stage)
        self.ycoord = self.mmc.getYPosition(self.stage)
        self.zcoord = self.mmc.getPosition()

            # Updating stage position every some interval

        self.coordinates = QLineEdit()
        self.time_now = time.time()*1000
        self.coordinates.setText(str(self.xcoord) + ',' + str(self.ycoord) + ','  + str(self.zcoord))
        self.timer = QTimer()
        self.timer.setInterval(20)
        self.timer.timeout.connect(self.update_all)
        self.timer.start()

            # Setting stage movespeed for control via app
        

        self.stagespeedbutton = QComboBox()
        self.stagefast   =   1000  # 1000um per click
        self.stagemedium =   100   # 100um per click
        self.stageslow   =   10    # 10um per click
        self.stagespeedbutton.addItems([str(self.stageslow), str(self.stagemedium), str(self.stagefast)])

        self.stage_speed = self.stagefast
        self.stagespeedbutton.currentTextChanged.connect(self.Set_stage_speed)
        #print(self.xcoord, self.ycoord, self.zcoord)

            # Stage movement buttons on app

        self.Xplus = QPushButton("X+")
        self.Xplus.clicked.connect(lambda: self.mmc.setXYPosition((self.xcoord + self.stage_speed), self.ycoord))

        self.Xminus = QPushButton("X-")
        self.Xminus.clicked.connect(lambda: self.mmc.setXYPosition((self.xcoord - self.stage_speed), self.ycoord))

        self.Yplus = QPushButton("Y+")
        self.Yplus.clicked.connect(lambda: self.mmc.setXYPosition(self.xcoord, (self.ycoord + self.stage_speed)))

        self.Yminus = QPushButton("Y-")
        self.Yminus.clicked.connect(lambda: self.mmc.setXYPosition(self.xcoord, (self.ycoord - self.stage_speed)))

        self.Zplus = QPushButton("Z+")
        self.Zplus.clicked.connect(lambda: self.mmc.setPosition((self.zcoord + self.stage_speed)))

        self.Zminus = QPushButton("Z-")
        self.Zminus.clicked.connect(lambda: self.mmc.setPosition((self.zcoord - self.stage_speed)))


        # Adding all stage widgets
        """self.layout.addWidget(self.coordinates)
        self.layout.addWidget(self.stagespeedbutton)
        self.layout.addWidget(self.Xplus)
        self.layout.addWidget(self.Xminus)
        self.layout.addWidget(self.Yplus)
        self.layout.addWidget(self.Yminus)
        self.layout.addWidget(self.Zplus)
        self.layout.addWidget(self.Zminus)"""



                                                            # Pipeline to display/save images
        
        self.snap_Button = QPushButton("Snap Image")
        self.snap_Button.clicked.connect(self.snap_image)
        self.image_Live = QLabel('No image loaded', self)
        
        self.live_Button = QToggleButton("Live Image")
        self.live_Button.clicked.connect(self.startstoplive_imaging)

         # create the video capture thread
        
        # start the thread
        # self.thread.start()

        # Create a button to load the image
        self.save_Button = QPushButton('Save Image', self)
        self.save_Button.clicked.connect(self.save_image)




        #Add buttons to layout
        """self.layout.addWidget(self.snap_Button)
        self.layout.addWidget(self.image_Live)
        self.layout.addWidget(self.save_Button)
        self.layout.addWidget(self.live_Button)"""

        self.layout.addWidget(self.snap_Button, row, 0)
        self.layout.addWidget(self.live_Button, row, 1)
        self.layout.addWidget(self.save_Button, row, 2)
        self.layout.addWidget(self.image_Live, row + 1, 0, 1, 3)

        self.layout.addWidget(self.coordinates, row + 2, 0, 1, 3)
        self.layout.addWidget(self.stagespeedbutton, row + 3, 0, 1, 3)

        self.layout.addWidget(self.Xplus, row + 4, 0)
        self.layout.addWidget(self.Xminus, row + 4, 1)
        self.layout.addWidget(self.Yplus, row + 5, 0)
        self.layout.addWidget(self.Yminus, row + 5, 1)
        self.layout.addWidget(self.Zplus, row + 6, 0)
        self.layout.addWidget(self.Zminus, row + 6, 1)

        self.layout.addWidget(self.camerapathlight, row + 7, 0)
        self.layout.addWidget(self.eyepathlight, row + 7, 2)



        # Finalizing GUI
        
        self.setWindowTitle('Microscope Control')
        self.show()
        

                                                # All functions for controlling the microscope go here


    def startstoplive_imaging(self):
        if self.live_Button.isChecked() == True:
            self.thread = VideoThread()
        # connect its signal to the update_image slot
            self.thread.change_pixmap_signal.connect(self.update_image)
            # start the thread
            self.thread.start()
        else:
            self.thread.stop()
        
    def snap_image(self):
        
        self.image_path = "C:\\Users\\Cell Culture Scope\\Pictures\\image.png" # Your image path
        # set(self.DIAshutterlight,'Value',1)
        #self.mmc.setConfig('TimeLapseJulia','DIA') # Load config
        self.mmc.setExposure(self.camera, 100) # Load Exposure
        self.mmc.setProperty(self.camera,'Gain', 10) # Load gain
        self.mmc.setProperty(self.lightpath,'Label',self.camerapath)
        #self.mmc.setProperty(self.DIAlamp, 'State', 1)
        #self.mmc.setProperty(self.DIAlamp, 'Intensity', 4)
        # self.mmc.setProperty(self.camera,'Exposure',self.num_expotimelapse_dia);
        #self.mmc.waitForConfig('TimeLapse','DIA')
        self.mmc.waitForDevice(self.filter) # Kinda custom wait for the slowest device that would allow snapping an image with everything ok
        self.mmc.waitForSystem()

        self.mmc.snapImage(); # Snap Image
        rawImage = self.mmc.getImage() # Process Image
        image_width = self.mmc.getImageWidth()
        image_height = self.mmc.getImageHeight()
        print(type(self.mmc.getBytesPerPixel))
        #if int(self.mmc.getBytesPerPixel) == 2:
        pixelType = np.uint16
        #else:
            #pixelType = np.uint8

        # Assuming rawImage is a byte array
        rawImage = np.frombuffer(rawImage, dtype= pixelType)
        rawImage = rawImage.reshape((image_height, image_width)).T
        # Assuming rawImage is a numpy array
        self.adjusted_image = exposure.rescale_intensity(rawImage)
        self.fake_save_image(self.adjusted_image) #Hard save image before display to compare
        raw_data = self.adjusted_image.tobytes()
        self.myQImage = QImage(raw_data, image_width, image_height, QImage.Format_Grayscale16)
        self.final_Image = QPixmap.fromImage(self.myQImage)
        self.image_Live.setPixmap(self.final_Image)
        #self.mmc.setProperty(self.DIAlamp, 'State', 0) # Lamp takes time to turn ON
    
    def save_image(self, save_path):
        if self.image_Live.pixmap():
            self.image_Live.pixmap().save(self.image_path)
            print(f"Image saved to {self.image_path}")
        else:
            print("No image to save")
    

    def fake_save_image(self, fed_image):
        image = Image.fromarray(fed_image, mode='I;16')

        # Save as PNG
        image.save('C:\\Users\\Cell Culture Scope\\Pictures\\image2.png')
         
    """def live_imaging(self):
        if self.live_Button.isChecked() == True:
            self.live_Button.setStyleSheet("background-color: #2ac555;")
            self.mmc.setProperty(self.DIAshutter,'State',0)
            self.mmc.setProperty(self.EPIshutter,'State',0)
            self.mmc.setProperty(self.lightpath,'Label',self.camerapath)
            self.mmc.initializeCircularBuffer()
            self.mmc.prepareSequenceAcquisition(self.camera)
            self.mmc.waitForDevice(self.DIAshutter)
            self.mmc.waitForDevice(self.camera)
            interval = 100 # image per X ms
            self.mmc.startContinuousSequenceAcquisition(interval)
            self.Sequencing = self.mmc.isSequenceRunning()
            while self.mmc.getRemainingImageCount() > 0 or self.Sequencing:
                if self.mmc.getRemainingImageCount() > 0:
                    liveimage = self.mmc.getLastImage()  #Retrieves last image taken
                    image_width = self.mmc.getImageWidth()
                    image_height = self.mmc.getImageHeight()
                    pixelType = np.uint16
                    #else:
                    #pixelType = np.uint8

                    # Assuming rawImage is a byte array
                    rawImage = np.frombuffer(liveimage, dtype= pixelType)
                    rawImage = rawImage.reshape((image_height, image_width)).T
                    # Assuming rawImage is a numpy array
                    self.adjusted_image = exposure.rescale_intensity(rawImage)
                    self.fake_save_image(self.adjusted_image) #Hard save image before display to compare
                    raw_data = self.adjusted_image.tobytes()
                    self.myQImage = QImage(raw_data, image_width, image_height, QImage.Format_Grayscale16)
                    self.final_Image = QPixmap.fromImage(self.myQImage)
                    self.image_Live.setPixmap(self.final_Image)

        else:
            self.mmc.stopSequenceAcquisition(self.camera) #Stop acquisition
            self.mmc.clearCircularBuffer()
            self.live_Button.setStyleSheet("background-color: #bb283a;")"""
             
    @pyqtSlot(np.ndarray)
    def update_image(self, cv_img):
        """Updates the image_label with a new opencv image"""
        qt_img = self.convert_np_qt(cv_img)
        self.image_Live.setPixmap(qt_img)

    def convert_np_qt(self, np_img):
        """Convert from an np image to QPixmap"""
        h, w = np_img.shape
        raw_data = np_img.tobytes()
        self.myQImage = QImage(raw_data, w, h, QImage.Format_Grayscale16)
        return QPixmap.fromImage(self.myQImage)

    def DIAlamp_ON(self):
         if self.dialamponlight.isChecked() == True:
              self.mmc.setProperty(self.DIAlamp, 'State', 1)
              self.dialamponlight.setStyleSheet("background-color: #2ac555;")
         else:
              self.mmc.setProperty(self.DIAlamp, 'State', 0)
              self.dialamponlight.setStyleSheet("background-color: #bb283a;")
    
    def PathtoCamera(self):
            self.mmc.setProperty(self.lightpath, 'Label', self.camerapath)
        
    def Set_zoom(self,item):
        print(item)
        self.mmc.setProperty(self.zoom, 'Label', item)
    
    def change_filter(self, filter_name):
         chosen_filter = [int(num) for num in re.findall(r'\d+', filter_name)]
         self.mmc.setProperty(self.filter, 'State', chosen_filter[0] - 1)
    
    def Set_stage_speed(self, item):
         self.stage_speed = int(item)
    
    def toggle_DIA_shutter(self):
         if self.DIAshutterbutton.isChecked() == True:
              self.mmc.setProperty(self.DIAshutter, 'State', 1)
              self.DIAshutterbutton.setStyleSheet("background-color: #2ac555;")
         else:
              self.mmc.setProperty(self.DIAshutter, 'State', 0)
              self.DIAshutterbutton.setStyleSheet("background-color: #bb283a;")
    
    def toggle_EPI_shutter(self):
         if self.EPIshutterbutton.isChecked() == True:
              self.mmc.setProperty(self.EPIshutter, 'State', 1)
              self.EPIshutterbutton.setStyleSheet("background-color: #2ac555;")
         else:
              self.mmc.setProperty(self.EPIshutter, 'State', 0)
              self.EPIshutterbutton.setStyleSheet("background-color: #bb283a;")
        
    
    def update_all(self):
        self.xcoord = self.mmc.getXPosition(self.stage)
        self.ycoord = self.mmc.getYPosition(self.stage)
        self.zcoord = self.mmc.getPosition()
        self.coordinates.setText(str(self.xcoord) + ',' + str(self.ycoord) + ','  + str(self.zcoord))

    

"""def main():
    app = QApplication(sys.argv)
    ex = MicroscopeControlGUI()
    sys.exit(app.exec_())

if __name__ == '__main__':
    main()"""

if __name__ == '__main__':
    import sys
    app = QtWidgets.QApplication(sys.argv)
    app.setStyle('Fusion')
    window = MicroscopeControlGUI()
    window.show()
    sys.exit(app.exec_())


<class 'builtin_function_or_method'>
<class 'builtin_function_or_method'>


SystemExit: 0

c:\ProgramData\anaconda3\envs\bootcamp_20240819\Lib\site-packages\IPython\core\interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
